In [2]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# 1. Read the dataset
df = pd.read_csv("Churn.csv")


# 2. Separate features and target
X = df.drop(columns=["Exited", "RowNumber", "CustomerId", "Surname"])
y = df["Exited"]


# 3. Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Identify categorical and numerical columns
categorical_columns = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()


# 4. Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_columns)
    ]
)


# 5. Create baseline and improved models
models = {

    "Baseline": MLPClassifier(
        hidden_layer_sizes=(32,),
        max_iter=500,
        random_state=42
    ),

    "Improved": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        alpha=0.001,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        max_iter=700,
        random_state=42
    )
}


# 6. Train and evaluate models
for name, classifier in models.items():

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", classifier)
    ])

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    print("\n", name, "model")

    print("Accuracy:",
          accuracy_score(y_test, predictions))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))


 Baseline model
Accuracy: 0.866
Confusion Matrix:
[[1530   63]
 [ 205  202]]

 Improved model
Accuracy: 0.8635
Confusion Matrix:
[[1507   86]
 [ 187  220]]
